# FedAS Client

>

In [ ]:
#| default_exp clients.fedas

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import copy
import random
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from fedai.clients.base_client import BaseClient
from fedai.utils.registery import AlgorithmRegistry
from torch.autograd import grad


<torch._C.Generator>

In [ ]:
#| export
@AlgorithmRegistry.register_client("fedas")
class ClientFedAS(BaseClient):
    def __init__(self,
                 id, # Unique identifier for the client
                 cfg, # A configuration object containing all the necessary parameters for training and evaluation.
                 train_loader, 
                 test_loader,
                 state, # A dictionary containing the model, optimizer and any changing component needed.
                 criterion,
                 device= None,
                 t= 0,
                 **kwargs # extra client-specific parameters (that cannot be passed in state and cfg), can be passed as here.
                 ):  
                 
        super().__init__(id, cfg, train_loader, test_loader, state, criterion, device, t, **kwargs)
        

In [ ]:
#| export
@patch
def set_parameters(self: ClientFedAS, model: nn.Module):
    local_prototypes = [[] for _ in range(self.cfg.data.num_classes)]
    self.model.to(self.device)
    model.to(self.device)
    # print(f'client{id}')
    for batch in self.train_loader:
        batch = self.send_to_device(batch)
        X, y = batch[self.data_key], batch[self.label_key]

        with torch.no_grad():
            proto_batch = self.model.backbone(X)

        # Scatter the prototypes based on their labels
        for proto, y in zip(proto_batch, y):
            local_prototypes[y.item()].append(proto)

    mean_prototypes = []

    # print(f'client{self.id}')
    for class_prototypes in local_prototypes:

        if not class_prototypes == []:
            # Stack the tensors for the current class
            stacked_protos = torch.stack(class_prototypes)

            # Compute the mean tensor for the current class
            mean_proto = torch.mean(stacked_protos, dim=0)
            mean_prototypes.append(mean_proto)
        else:
            mean_prototypes.append(None)

    # Align global model's prototype with the local prototype
    alignment_optimizer = torch.optim.SGD(model.backbone.parameters(), lr=0.01)  # Adjust learning rate and optimizer as needed
    alignment_loss_fn = torch.nn.MSELoss()

    # print(f'client{self.id}')
    for _ in range(1):  # Iterate for 1 epochs; adjust as needed
        for batch in self.train_loader:
            batch = self.send_to_device(batch)
            X, y = batch[self.data_key], batch[self.label_key]
            global_proto_batch = model.backbone(X)
            loss = 0
            total_samples = len(y)
            for label in y.unique():
                if mean_prototypes[label.item()] is not None:
                    class_samples = (y == label).sum()
                    target = mean_prototypes[label.item()].unsqueeze(0) # Shape becomes [1, feature_dim]
                    class_loss = alignment_loss_fn(global_proto_batch[y == label], target.expand_as(global_proto_batch[y == label]))
                    # class_loss = alignment_loss_fn(global_proto_batch[y == label], mean_prototypes[label.item()])
                    loss += class_loss * (class_samples / total_samples)
            alignment_optimizer.zero_grad()
            loss.backward()
            alignment_optimizer.step()

    # Substitute the parameters of the base, enabling personalization
    for new_param, old_param in zip(model.backbone.parameters(), self.model.backbone.parameters()):
        old_param.data = new_param.data.clone()


    # end



### Training

In [ ]:
#| export
@patch
def train_step(self: ClientFedAS):
    self.model.to(self.device)
    self.model.train()
    
    self.model.to(self.device)
    self.model.train()
    for _ in range(self.cfg.local_epochs):
        for i, batch in enumerate(self.train_loader):
            self.optimizer.zero_grad()

            batch = self.send_to_device(batch)
            X, y = batch[self.data_key], batch[self.label_key]
            outputs = self.model(X)
            
            loss = self.criterion(outputs, y)
            loss.backward()
            self.optimizer.step()


In [ ]:
#| export
@patch
def fit(self: ClientFedAS, active: bool):
    
    self.model.to(self.device)
    if active:
        self.train_step()

    self.model.eval()
    # Compute FIM and its trace after training
    fim_trace_sum = 0
    for i, batch in enumerate(self.train_loader):
        # Forward pass
        batch = self.send_to_device(batch)
        X, y = batch[self.data_key], batch[self.label_key]
        outputs = self.model(X)
        # Negative log likelihood as our loss
        nll = -torch.nn.functional.log_softmax(outputs, dim=1)[range(len(y)), y].mean()

        # Compute gradient of the negative log likelihood w.r.t. model parameters
        grads = grad(nll, self.model.parameters())

        # Compute and accumulate the trace of the Fisher Information Matrix
        for g in grads:
            fim_trace_sum += torch.sum(g ** 2).detach()

    # add the fisher log
    self.fim_trace_history.append(fim_trace_sum.item())


### Saving State

In [ ]:
#| export
@patch
def save_state(self: ClientFedAS, save_to_disk= False):  

    state_dict = {}
    state_dict['model'] = self.model.state_dict()
    state_dict['optimizer'] = self.optimizer.state_dict()
    state_dict['fim_trace_history'] = self.fim_trace_history

    if save_to_disk:
        state_path = os.path.join(self.cfg.server.save_dir, str(self.t), f"local_output_{self.id}", "state.pth")
        if not os.path.exists(os.path.dirname(state_path)):
            os.makedirs(os.path.dirname(state_path))

        torch.save(state_dict, state_path)

    return state_dict

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()